<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/AI_Adventure_Game_Mistral_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AI Explorer Adventure Game — H2E Edition (Mistral 4-bit, HuggingFace)
### Governed by H2E Sheriff · M1 Metric · Geodesic Safety on H² × SPD(3)
**Reference:** Morales Aguilera, F. — Sovereign Machine Lab (SOMALA), Montreal · IEEE 2026

**Fully open-source — no API keys required for the LLM.**
Mistral-7B-Instruct is loaded directly from HuggingFace and quantized to 4-bit on GPU.

**Requirements:**
- Google Colab with GPU runtime (T4 or better — Runtime → Change runtime type → GPU)
- HuggingFace token in Colab Secrets as `HF_TOKEN` (needed to download Mistral weights)

**Two cells:**
- **Cell 1** — Install packages, load Mistral-7B-Instruct-v0.3 in 4-bit NF4
- **Cell 2** — Full game with H2E Sheriff M1 safety gate

---
**Configuration (edit in Cell 2 before running):**
- `TOTAL_GAME_TURNS` — 5 = quick demo | 20 = standard | 50–100 = full experience
- `PLAYER_AGE_GROUP` — `'younger'` (6–9) or `'older'` (10–14)
- `LANGUAGE` — `'english'` or `'spanish'`

In [1]:
!pip install transformers torch accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.7 MB/s eta 0:00:00


In [ ]:
# Cell 1: LOAD AND COMPRESS MODEL TO 4-BIT - RUN THIS FIRST
print("="*60)
print("LOADING MISTRAL MODEL (4-bit compressed)")
print("="*60)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

# 4-bit compression config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading and compressing model to 4-bit (5-10 min first time)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("\nMODEL LOADED in 4-bit!")
print(f"Memory usage: ~3-4 GB VRAM")
print("Run Cell 2 to play the game")
print("="*60)


In [8]:
import IPython.display as display
from google.colab import output
import json
import re
import asyncio
import nest_asyncio
import numpy as np
import hashlib
from scipy.linalg import logm
from concurrent.futures import ThreadPoolExecutor

nest_asyncio.apply()
_executor = ThreadPoolExecutor(max_workers=1)

# ============================================================
# CONFIGURATION — edit before running
# ============================================================
TOTAL_GAME_TURNS    = 5
PLAYER_AGE_GROUP    = 'younger'   # 'younger' (6-9) or 'older' (10-14)
LANGUAGE            = 'spanish'   # 'english' or 'spanish'
STREAK_BONUS_POINTS = 5
MAX_NEW_TOKENS      = 1200
TEMPERATURE         = 0.7
# ============================================================

POINTS_PER_CORRECT_ANSWER = 100 / TOTAL_GAME_TURNS

# ============================================================
# H2E SHERIFF — M1 METRIC
# Geodesic safety on H2 x SPD(3)
# Reference: Morales Aguilera, IEEE 2026
# doi:10.5281/zenodo.19972045
# ============================================================
def compute_lambda():
    primes = [2, 3, 5, 7, 11, 13]
    I = 1.0
    for p in primes:
        I *= (1 - p ** (-0.5))
    K = 44.601732
    return I * K

H2E_LAMBDA = compute_lambda()
print(f'H2E Sheriff initialised. Lambda = {H2E_LAMBDA:.6f} (from primes 2,3,5,7,11,13)')

_P0_REF_TEXT = 'what is artificial intelligence how do computers learn from data'

def _text_to_features(text):
    t = text.lower().strip() if text else ''
    n = max(len(t), 1)
    return np.array([
        min(len(t), 200) / 200.0,
        sum(c in 'aeiou' for c in t) / n,
        sum(c.isdigit() for c in t) / n,
        sum(not c.isalnum() and not c.isspace() for c in t) / n,
        min(len(t.split()), 50) / 50.0,
        sum(c.isupper() for c in text) / max(len(text), 1) if text else 0.0,
    ])

def _features_to_H2(f):
    return np.array([f[0] * 2.0 - 1.0, max(f[1] + f[4] + 0.1, 0.01)])

def _features_to_SPD3(f):
    v = f[:3] + 0.1
    B = np.diag(v) + 0.01 * np.outer(f[:3], f[3:6])
    return B @ B.T + 0.1 * np.eye(3)

def _hyperbolic_distance(p1, p2):
    x1, y1 = p1
    x2, y2 = p2
    return np.arccosh(max(1.0 + ((x1 - x2)**2 + (y1 - y2)**2) / (2.0 * y1 * y2), 1.0))

def _spd_distance(A, B):
    try:
        ic = np.linalg.inv(np.linalg.cholesky(A))
        return float(np.linalg.norm(logm(ic @ B @ ic.T), 'fro'))
    except Exception:
        return 0.0

_p0_f    = _text_to_features(_P0_REF_TEXT)
_P0_H2   = _features_to_H2(_p0_f)
_P0_SPD3 = _features_to_SPD3(_p0_f)

def m1_evaluate(text):
    f     = _text_to_features(text)
    d_h2  = _hyperbolic_distance(_P0_H2, _features_to_H2(f))
    d_spd = _spd_distance(_P0_SPD3, _features_to_SPD3(f))
    d_M   = float(np.sqrt(d_h2**2 + d_spd**2))
    sroi  = float(np.exp(-d_M / 50.0))
    dec   = 'ACCEPT' if sroi >= H2E_LAMBDA else 'REJECT'
    data  = f'{text}|{dec}|{sroi:.6f}|{H2E_LAMBDA:.6f}'
    h     = hashlib.sha256(data.encode()).hexdigest()[:16]
    return sroi, d_M, d_h2, d_spd, dec, h

# ============================================================
# AI CONCEPT LIBRARY
# ============================================================
ALL_AI_CONCEPTS = [
    'What is Artificial Intelligence?',
    'The difference between AI and regular programs',
    'How computers store information (memory)',
    'Binary code — how computers speak in 0s and 1s',
    'What is an algorithm?',
    'What is a dataset?',
    'What is a model in AI?',
    'Machine Learning — learning from examples',
    'Supervised learning — learning with a teacher',
    'Unsupervised learning — finding patterns alone',
    'Reinforcement learning — learning by trial and error',
    'Training data vs testing data',
    'Overfitting — when AI memorises instead of learns',
    'Features — what the AI pays attention to',
    'Labels — teaching AI the right answers',
    'Neural networks — inspired by the human brain',
    'Neurons and connections',
    'Deep learning — many layers of thinking',
    'How AI gets better with practice (gradient descent)',
    'Computer Vision — AI that sees',
    'Image recognition — identifying objects in photos',
    'Face recognition',
    'Object detection — finding things in images',
    'How cameras and pixels work',
    'Medical imaging AI (detecting illness in scans)',
    'Natural Language Processing — AI that reads',
    'Text classification — sorting messages by topic',
    'Sentiment analysis — detecting happy or sad text',
    'Machine translation — translating languages',
    'Chatbots and virtual assistants',
    'Speech recognition — AI that listens',
    'Text-to-speech — AI that talks',
    'Large Language Models like GPT and Mistral',
    'Tokens — how AI breaks text into chunks',
    'Robots and automation',
    'Self-driving cars',
    'Drones and autonomous flight',
    'Robot arms in factories',
    'Sensors — how robots feel the world',
    'Decision trees — AI playing 20 questions',
    'Probability — measuring how likely something is',
    'Recommendation systems (like YouTube suggestions)',
    'Search engines',
    'Spam filters',
    'Fraud detection',
    'AI in video games (NPCs)',
    'Game-playing AI (Chess, Go, AlphaGo)',
    'Generative AI — creating art and music',
    'AI bias — when AI is unfair',
    'Privacy and data',
    'AI helping doctors',
    'AI helping the environment',
    'AI safety',
    'H2E Sheriff — a real computer safety system using mathematics',
    'SROI — Safety Return on Investment: a safety score from 0 to 1',
    'Geodesic distance on a curved mathematical surface',
    'Deterministic AI — always gives the same answer for the same input',
    'Prime numbers and AI safety threshold Lambda',
    'Audit hash — a SHA256 fingerprint of an AI decision',
    'Hard-stop — AI blocks an action when safety score is too low',
]

# ============================================================
# GLOBAL GAME STATE
# ============================================================
current_story_description = 'Welcome! Click Start New AI Adventure to begin.'
current_choices           = []
game_over_status          = False
error_message             = ''
info_message              = ''
player_score              = 0
turn_counter              = 0
streak_counter            = 0
covered_concepts          = []
concepts_log              = []
last_sroi                 = 1.0
last_audit_hash           = ''
last_decision             = 'ACCEPT'
total_rejections          = 0

# ============================================================
# HIGH SCORES
# ============================================================
try:
    import os
    import pandas as pd
    game_data_dir    = '/content/drive/My Drive/AIExplorerGameData'
    os.makedirs(game_data_dir, exist_ok=True)
    HIGH_SCORES_FILE = os.path.join(game_data_dir, 'high_scores_mistral.csv')

    def load_high_scores_from_drive():
        if not os.path.exists(HIGH_SCORES_FILE):
            return []
        try:
            df = pd.read_csv(HIGH_SCORES_FILE)
            if 'player_name' in df.columns and 'score' in df.columns:
                df['score'] = pd.to_numeric(df['score'], errors='coerce')
                df.dropna(subset=['score'], inplace=True)
                df['score'] = df['score'].astype(int)
                return list(df.sort_values('score', ascending=False)[
                    ['player_name', 'score']].itertuples(index=False, name=None))
        except Exception as e:
            print(f'Score load error: {e}')
        return []

    def save_high_scores_to_drive(scores_list):
        try:
            pd.DataFrame(scores_list, columns=['player_name', 'score']).to_csv(
                HIGH_SCORES_FILE, index=False)
        except Exception as e:
            print(f'Score save error: {e}')

    get_high_scores = load_high_scores_from_drive

except Exception:
    _mem_scores = []

    def load_high_scores_from_drive():
        return sorted(_mem_scores, key=lambda x: x[1], reverse=True)

    def save_high_scores_to_drive(scores_list):
        global _mem_scores
        _mem_scores = scores_list

    get_high_scores = load_high_scores_from_drive

# ============================================================
# JSON EXTRACTION — robust parser for local model output
# ============================================================
def extract_json(text):
    # 1. Direct parse
    try:
        return json.loads(text.strip())
    except Exception:
        pass
    # 2. Markdown code block
    for pat in [r'```json\s*([\s\S]*?)\s*```', r'```\s*([\s\S]*?)\s*```']:
        m = re.search(pat, text)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                pass
    # 3. Find outermost { ... }
    start = text.find('{')
    if start != -1:
        depth = 0
        for i, c in enumerate(text[start:], start):
            if c == '{':
                depth += 1
            elif c == '}':
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i + 1])
                    except Exception:
                        break
    return None

# ============================================================
# PROMPTS — exact originals from Gemini version
# JSON_FORMAT appended to each prompt so Mistral knows the output format
# ============================================================
JSON_FORMAT = (
    'Respond with ONLY a valid JSON object, no other text:\n'
    '{"description":"<story text>","choices":["<c1>","<c2>","<c3>"]'
    ',"gameover":false,"score_change":<int>,"concept_name":"<name>"'
    ',"concept_explained":"<one sentence>","correct_answer_feedback":"<hint or empty>"}'
)

def _age_instruction():
    if PLAYER_AGE_GROUP == 'younger':
        return 'Use very simple words suitable for age 6-9. Short sentences. Fun emojis. No jargon.'
    return 'Use clear language for age 10-14. Slightly technical terms are fine if explained simply.'

def _language_instruction():
    return 'Respond entirely in Spanish.' if LANGUAGE == 'spanish' else 'Respond in English.'

def _available_concepts():
    remaining = [c for c in ALL_AI_CONCEPTS if c not in covered_concepts]
    return remaining[:20] if remaining else ALL_AI_CONCEPTS

def _no_fantasy_rules():
    return """CRITICAL RULES — MUST FOLLOW EXACTLY:
- Teach REAL science. Absolutely NO fantasy, magic, sparkles, fairy tales, lassos, or invented powers.
- H2E Sheriff is a REAL computer safety program built by a scientist. It runs on a computer and calculates a number called SROI to decide if an action is safe.
- Never describe H2E Sheriff as a cowboy, wizard, magical creature, or fantasy character. It is real software.
- SROI is a real mathematical score from 0 to 1. The threshold Lambda is computed from prime numbers {2,3,5,7,11,13}.
- Actions with SROI above Lambda are accepted. Actions below Lambda are blocked immediately — hard stop.
- Keep the story in a real-world setting: AI research lab, school, university, real city.
- Use simple words appropriate for children — but keep the science honest and grounded in reality."""

def initial_game_prompt():
    return (
        f"You are the game master for a text adventure teaching REAL Artificial Intelligence concepts to children aged 8-12.\n"
        f"{_age_instruction()}\n"
        f"{_language_instruction()}\n"
        f"\n"
        f"{_no_fantasy_rules()}\n"
        f"\n"
        "Opening scenario: A child visits an AI research lab with a scientist (parent or teacher). "
        "On a computer screen the scientist shows the H2E Sheriff — a real safety program that monitors every action in the lab. "
        "The scientist explains: H2E Sheriff computes a safety score called SROI (a number from 0 to 1) for every action. "
        "If SROI is above the threshold Lambda, the action proceeds. "
        "If it is below Lambda, the system immediately blocks it with a hard stop — no exceptions. "
        "The child is curious and wants to learn more about AI.\n"
        "Description: 100-200 words. Provide 3 realistic choices the child can make in the lab.\n"
        + JSON_FORMAT +
        "\nSet score_change=0, concept_name to empty string, concept_explained to empty string, correct_answer_feedback to empty string."
    )

def action_prompt(player_action):
    already_taught = ', '.join(covered_concepts[-10:]) if covered_concepts else 'none yet'
    available      = _available_concepts()
    progress_pct   = int((turn_counter / TOTAL_GAME_TURNS) * 100)
    if progress_pct < 25:
        complexity = 'very basic — simple analogies only'
    elif progress_pct < 60:
        complexity = 'moderate — build on earlier concepts with a bit more detail'
    else:
        complexity = 'deeper — connect multiple concepts and add nuance'
    return (
        f"You are the game master for a text adventure teaching REAL Artificial Intelligence concepts to children aged 8-12.\n"
        f"{_age_instruction()}\n"
        f"{_language_instruction()}\n"
        f"\n"
        f"{_no_fantasy_rules()}\n"
        f"\n"
        f'The player chose: "{player_action}".\n'
        f"This is turn {turn_counter} of {TOTAL_GAME_TURNS} ({progress_pct}% through the game).\n"
        f"Current streak: {streak_counter} correct answers in a row.\n"
        f"\n"
        "When relevant, mention H2E Sheriff as a real computer safety system running in the background of the lab — not as a character or creature.\n"
        f"\n"
        f"CONCEPTS ALREADY TAUGHT — DO NOT REPEAT THESE: {already_taught}\n"
        f"\n"
        f"Choose ONE new concept from this list for this turn:\n{json.dumps(available, indent=2)}\n"
        f"\n"
        f"Concept complexity level: {complexity}.\n"
        f"\n"
        "Instructions:\n"
        f"- Write a description (100-180 words) advancing the story and teaching the chosen concept.\n"
        "- Provide exactly 3 realistic choices. Exactly ONE is the most correct for learning AI.\n"
        f"- If the player's last action was the best choice: score_change = {int(POINTS_PER_CORRECT_ANSWER)}. Otherwise: score_change = 0.\n"
        "- If score_change is 0: provide a warm hint in correct_answer_feedback. If > 0: leave correct_answer_feedback empty.\n"
        "- concept_name = short name (e.g. Machine Learning).\n"
        "- concept_explained = one child-friendly sentence explaining the concept accurately.\n"
        "- Do NOT set gameover to true.\n"
        + JSON_FORMAT
    )

def end_game_prompt():
    concepts_summary = ', '.join([c[1] for c in concepts_log]) if concepts_log else 'many AI concepts'
    return (
        f"You are the game master. The adventure is now complete after {TOTAL_GAME_TURNS} turns.\n"
        f"{_age_instruction()}\n"
        f"{_language_instruction()}\n"
        f"\n"
        f"{_no_fantasy_rules()}\n"
        f"\n"
        f"The player earned {player_score} points and learned about: {concepts_summary}.\n"
        "The H2E Sheriff safety system protected every step of their journey through the lab.\n"
        "Write a warm, exciting 80-120 word closing paragraph set in the real lab, celebrating what the child discovered.\n"
        "Reference H2E Sheriff as the safety program that kept every action verified and safe.\n"
        + JSON_FORMAT +
        "\nSet gameover=true, choices=[], score_change=0, concept_name to empty string, concept_explained to empty string, correct_answer_feedback to empty string."
    )

def minimal_fallback_prompt():
    return (
        "You are a game master. Continue a short story set in an AI research lab.\n"
        "Respond with ONLY valid JSON, no other text:\n"
        '{"description":"<60 word story paragraph in the lab>"'
        ',"choices":["Look at the screen","Ask the scientist","Take notes"]'
        ',"gameover":false,"score_change":0'
        ',"concept_name":"What is Artificial Intelligence?"'
        ',"concept_explained":"AI is software that learns from data to make decisions."'
        ',"correct_answer_feedback":""}'
    )

# ============================================================
# LOCAL MISTRAL INFERENCE — stateless, { prefix forces JSON
# ============================================================
def _run_inference(prompt_text):
    import torch
    messages  = [{'role': 'user', 'content': prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Append '{' — forces the model to begin its response with a JSON object
    formatted = formatted + '{'

    inputs = tokenizer(
        formatted,
        return_tensors='pt',
        truncation=True,
        max_length=2048,
        add_special_tokens=False,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    prompt_len = inputs['input_ids'].shape[1]
    print(f'[Mistral] prompt_tokens={prompt_len}  max_new={MAX_NEW_TOKENS}')

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )

    new_ids  = output_ids[0][prompt_len:]
    response = tokenizer.decode(new_ids, skip_special_tokens=True)
    full     = '{' + response   # re-attach the prefix brace
    print(f'[Mistral raw] {full[:300]}')
    return full

# ============================================================
# LLM CALL — async, stateless, with one retry on parse failure
# ============================================================
async def call_llm(prompt_text):
    global error_message
    error_message = ''

    prompts = [prompt_text, minimal_fallback_prompt()]
    for attempt, prompt in enumerate(prompts):
        try:
            loop         = asyncio.get_event_loop()
            response_txt = await loop.run_in_executor(_executor, _run_inference, prompt)
            parsed       = extract_json(response_txt)

            if parsed is not None:
                parsed.setdefault('description',             'The adventure continues in the AI lab.')
                parsed.setdefault('choices',                 ['Look around', 'Ask a question', 'Take notes'])
                parsed.setdefault('gameover',                False)
                parsed.setdefault('score_change',            0)
                parsed.setdefault('concept_name',            '')
                parsed.setdefault('concept_explained',       '')
                parsed.setdefault('correct_answer_feedback', '')
                return parsed

            print(f'[JSON parse failed — attempt {attempt + 1}]')

        except Exception as e:
            print(f'[Inference error attempt {attempt + 1}]: {e}')
            error_message = f'Inference error: {e}'

    error_message = 'Could not generate valid JSON after 2 attempts. Please try again.'
    return None

# ============================================================
# GAME LOGIC
# ============================================================
async def start_new_game():
    global current_story_description, current_choices, game_over_status
    global error_message, info_message, player_score, turn_counter, streak_counter
    global covered_concepts, concepts_log, POINTS_PER_CORRECT_ANSWER
    global last_sroi, last_audit_hash, last_decision, total_rejections

    game_over_status          = False
    error_message             = ''
    info_message              = ''
    current_story_description = 'Generating your adventure with Mistral 4-bit...'
    current_choices           = []
    player_score              = 0
    turn_counter              = 0
    streak_counter            = 0
    covered_concepts          = []
    concepts_log              = []
    last_sroi                 = 1.0
    last_audit_hash           = ''
    last_decision             = 'ACCEPT'
    total_rejections          = 0
    POINTS_PER_CORRECT_ANSWER = 100 / TOTAL_GAME_TURNS

    update_game_ui(is_loading=True)
    scenario = await call_llm(initial_game_prompt())

    if scenario:
        current_story_description = scenario.get('description', '')
        current_choices           = scenario.get('choices', [])
        game_over_status          = scenario.get('gameover', False)
    else:
        current_story_description = 'Failed to start. Make sure Cell 1 ran successfully.'
        game_over_status          = True
        error_message             = 'Error: Could not generate opening scenario.'

    update_game_ui(is_loading=False)


async def handle_player_action(player_action):
    global current_story_description, current_choices, game_over_status
    global error_message, info_message, player_score, turn_counter, streak_counter
    global covered_concepts, concepts_log
    global last_sroi, last_audit_hash, last_decision, total_rejections

    if not player_action.strip():
        error_message = 'Please enter your action or click a choice button.'
        update_game_ui(is_loading=False)
        return

    if game_over_status:
        return

    # ── H2E SHERIFF GATE ──────────────────────────────────────
    sroi, d_M, d_h2, d_spd, decision, audit_hash = m1_evaluate(player_action)
    last_sroi       = sroi
    last_audit_hash = audit_hash
    last_decision   = decision
    print(f'[H2E Sheriff] "{player_action[:40]}" | SROI={sroi:.4f} | Lambda={H2E_LAMBDA:.4f} | {decision} | Hash={audit_hash}')

    if decision == 'REJECT':
        total_rejections += 1
        error_message = (
            f'H2E Sheriff HARD STOP — SROI {sroi:.4f} < Lambda {H2E_LAMBDA:.4f}. '
            'Action blocked. Try a different input!'
        )
        update_game_ui(is_loading=False)
        return
    # ── END SHERIFF GATE ──────────────────────────────────────

    turn_counter += 1
    current_story_description += f'\n\nYou chose: "{player_action}"'
    update_game_ui(is_loading=True)

    # Final turn — score the last action first, then generate closing narrative
    if turn_counter >= TOTAL_GAME_TURNS:
        # Step 1: score the last player action like any normal turn
        scenario = await call_llm(action_prompt(player_action))
        if scenario:
            score_change = scenario.get('score_change', 0)
            concept_name = scenario.get('concept_name', '')
            concept_exp  = scenario.get('concept_explained', '')
            feedback     = scenario.get('correct_answer_feedback', '')
            if concept_name and concept_name not in covered_concepts:
                covered_concepts.append(concept_name)
            if score_change > 0:
                streak_counter += 1
                bonus      = STREAK_BONUS_POINTS * (streak_counter - 1) if streak_counter > 1 else 0
                total_gain = score_change + bonus
                player_score += total_gain
                concepts_log.append((turn_counter, concept_name, total_gain))
            else:
                streak_counter = 0
                concepts_log.append((turn_counter, concept_name, 0))

        # Step 2: generate warm closing narrative
        closing = await call_llm(end_game_prompt())
        current_story_description = (
            closing.get('description', 'Adventure complete! Well done!')
            if closing else 'Adventure complete! Well done!'
        )
        current_choices  = []
        game_over_status = True
        info_message     = f'Game Over! Your final score is {round(player_score)}.'
        update_game_ui(is_loading=False)
        return

    scenario = await call_llm(action_prompt(player_action))

    if scenario:
        score_change = scenario.get('score_change', 0)
        concept_name = scenario.get('concept_name', '')
        concept_exp  = scenario.get('concept_explained', '')
        feedback     = scenario.get('correct_answer_feedback', '')

        if concept_name and concept_name not in covered_concepts:
            covered_concepts.append(concept_name)

        if score_change > 0:
            streak_counter += 1
            bonus      = STREAK_BONUS_POINTS * (streak_counter - 1) if streak_counter > 1 else 0
            total_gain = score_change + bonus
            player_score += total_gain
            concepts_log.append((turn_counter, concept_name, total_gain))
        else:
            streak_counter = 0
            total_gain     = 0
            concepts_log.append((turn_counter, concept_name, 0))

        description = scenario.get('description', '')
        if score_change > 0:
            streak_msg = f' {streak_counter} in a row!' if streak_counter > 1 else ''
            bonus_msg  = f' (+{int(bonus)} streak bonus)' if bonus > 0 else ''
            description += f'\n\nYou earned {int(total_gain)} points!{bonus_msg}{streak_msg}'
        elif feedback:
            description += f'\n\nNot quite! {feedback}'

        if concept_exp:
            description += f"\n\n<span class='ai-concept-highlight'>AI Concept: {concept_name} — {concept_exp}</span>"

        current_story_description = description
        current_choices           = scenario.get('choices', [])
    else:
        current_story_description = 'Something went wrong. Please try choosing again.'
        turn_counter -= 1
        error_message = 'Error: Could not generate response. Please try again.'

    update_game_ui(is_loading=False)


# ============================================================
# UI
# ============================================================
def update_game_ui(is_loading=False):
    global current_story_description, current_choices, game_over_status
    global error_message, info_message, player_score, turn_counter, TOTAL_GAME_TURNS
    global streak_counter, covered_concepts, concepts_log
    global last_sroi, last_audit_hash, last_decision, total_rejections

    progress_pct     = int((turn_counter / TOTAL_GAME_TURNS) * 100) if TOTAL_GAME_TURNS > 0 else 0
    choices_html     = ''
    input_area_html  = ''
    name_input_html  = ''
    high_scores_html = ''

    if not game_over_status and not is_loading:
        for i, choice in enumerate(current_choices):
            esc = choice.replace("'", "\\'")
            choices_html += (
                f'<button class="choice-button bg-blue-600 hover:bg-blue-800 text-white '
                f'font-bold py-2 px-4 rounded-full m-2 transition duration-300 ease-in-out '
                f'transform hover:scale-105 shadow-lg" '
                f'onclick="google.colab.kernel.invokeFunction('
                f"'handle_player_action', ['{esc}'], {{}})"
                f'">{i + 1}. {choice}</button>'
            )
        input_area_html = (
            '<div class="input-area">'
            '<input type="text" id="player-input" '
            'placeholder="Or type your own action here..." '
            'onkeydown="if(event.key===\'Enter\') '
            "google.colab.kernel.invokeFunction('handle_player_action',"
            "[document.getElementById('player-input').value],{}); "
            'return event.key!==\'Enter\';" '
            'class="rounded-lg border border-gray-600 focus:ring-blue-500 focus:border-blue-500">'
            '<button class="go-button bg-green-600 hover:bg-green-800 text-white font-bold '
            'py-2 px-6 rounded-full transition duration-300 ease-in-out transform '
            'hover:scale-105 shadow-lg" '
            "onclick=\"google.colab.kernel.invokeFunction('handle_player_action',"
            "[document.getElementById('player-input').value],{})\">"
            'Go!</button></div>'
        )

    if game_over_status and not is_loading:
        name_input_html = (
            f'<div class="input-area flex-col mt-4">'
            f'<p class="text-xl font-bold">Your final score is: '
            f'<span class="text-yellow-400">{round(player_score)}</span></p>'
            f'<p class="mt-2">Enter your name to save your score!</p>'
            f'<input type="text" id="player_name_input" placeholder="Your Name" '
            f'class="rounded-lg border border-gray-600 focus:ring-blue-500 '
            f'focus:border-blue-500 mb-2 mt-2">'
            f'<button class="save-button bg-green-600 hover:bg-green-800 text-white '
            f'font-bold py-2 px-6 rounded-full transition duration-300 ease-in-out '
            f'transform hover:scale-105 shadow-lg" '
            f'onclick="google.colab.kernel.invokeFunction(\'save_score_callback\','
            f'[document.getElementById(\'player_name_input\').value],{{}})">'
            f'Save Score</button></div>'
        )
        high_scores = get_high_scores()
        if high_scores:
            high_scores_html = (
                "<div class='high-scores-box mt-4 p-4 rounded-lg bg-gray-700 shadow-inner'>"
                "<h3 class='text-xl font-bold mb-2 text-blue-400'>Top 5 AI Explorers</h3>"
                "<ul class='list-none mx-auto w-fit text-left'>"
            )
            for name, score in high_scores[:5]:
                high_scores_html += (
                    f"<li class='py-1 border-b border-gray-600'>{name}: "
                    f"<span class='font-bold text-yellow-400'>{score}</span> points</li>"
                )
            high_scores_html += '</ul></div>'

    concepts_log_html = ''
    if concepts_log:
        concepts_log_html = (
            "<div class='concepts-log mt-4 p-3 rounded-lg bg-gray-700'>"
            "<h3 class='text-base font-bold mb-2 text-purple-400'>Concepts Learned</h3>"
            "<ul class='list-none text-left'>"
        )
        for t, c, pts in concepts_log[-10:]:
            icon = 'OK' if pts > 0 else '--'
            concepts_log_html += (
                f"<li class='py-1 border-b border-gray-600 text-sm'>"
                f"[{icon}] Turn {t}: "
                f"<span class='text-purple-300'>{c}</span> — "
                f"<span class='text-yellow-400'>{int(pts)} pts</span></li>"
            )
        concepts_log_html += '</ul></div>'

    streak_html    = f"<span class='text-orange-400 font-bold ml-2'>{streak_counter} streak!</span>" if streak_counter > 1 else ''
    sroi_pct       = int(last_sroi * 100)
    sroi_bar_color = '#22c55e' if last_decision == 'ACCEPT' else '#ef4444'
    dec_color      = 'text-green-400' if last_decision == 'ACCEPT' else 'text-red-400'

    sheriff_panel = f"""
    <div class="sheriff-panel mt-4 p-4 rounded-xl" style="background:rgba(15,23,42,0.8);border:1px solid #334155;">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;">
            <span style="color:#60a5fa;font-weight:700;font-size:0.9rem;">H2E Sheriff — M1 Metric</span>
            <span style="color:#94a3b8;font-size:0.75rem;">H2 x SPD(3) | Mistral 4-bit NF4</span>
        </div>
        <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;">
            <div style="flex:1;min-width:140px;">
                <div style="display:flex;justify-content:space-between;margin-bottom:3px;">
                    <span style="color:#94a3b8;font-size:0.75rem;">SROI = exp(-dM/50)</span>
                    <span style="color:#e2e8f0;font-size:0.75rem;font-weight:700;">{last_sroi:.4f}</span>
                </div>
                <div style="background:#1e293b;border-radius:999px;height:8px;overflow:hidden;">
                    <div style="height:100%;width:{sroi_pct}%;background:{sroi_bar_color};border-radius:999px;transition:width 0.5s;"></div>
                </div>
                <div style="display:flex;justify-content:space-between;margin-top:2px;">
                    <span style="color:#64748b;font-size:0.65rem;">0</span>
                    <span style="color:#f59e0b;font-size:0.65rem;">Lambda={H2E_LAMBDA:.4f}</span>
                    <span style="color:#64748b;font-size:0.65rem;">1</span>
                </div>
            </div>
            <div style="text-align:center;">
                <div class="{dec_color}" style="font-weight:900;font-size:1rem;">{last_decision}</div>
                <div style="color:#64748b;font-size:0.65rem;">decision</div>
            </div>
            <div style="text-align:center;">
                <div style="color:#a78bfa;font-weight:700;font-size:0.85rem;">{total_rejections}</div>
                <div style="color:#64748b;font-size:0.65rem;">rejections</div>
            </div>
            <div style="text-align:right;flex:1;">
                <div style="color:#475569;font-size:0.65rem;">SHA256 audit</div>
                <div style="color:#334155;font-size:0.6rem;font-family:monospace;">{last_audit_hash if last_audit_hash else "—"}</div>
            </div>
        </div>
        <div style="margin-top:6px;color:#475569;font-size:0.65rem;">
            Lambda computed from primes 2,3,5,7,11,13 — never hardcoded
        </div>
    </div>"""

    msg_class   = 'hidden'
    msg_content = ''
    if is_loading:
        msg_class   = 'bg-blue-900 text-blue-200 p-3 rounded-lg text-center'
        msg_content = 'Mistral 4-bit is thinking... Please wait.'
    elif error_message:
        msg_class   = 'bg-red-700 text-white p-3 rounded-lg text-center'
        msg_content = error_message
    elif info_message:
        msg_class   = 'bg-blue-700 text-white p-3 rounded-lg text-center'
        msg_content = info_message

    html_content = f"""
    <script src="https://cdn.tailwindcss.com"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;700&display=swap" rel="stylesheet">
    <style>
        body {{ font-family:'Inter',sans-serif; background:linear-gradient(135deg,#1f2937 0%,#374151 100%); color:#e5e7eb; display:flex; justify-content:center; align-items:center; min-height:100vh; margin:0; padding:20px; box-sizing:border-box; }}
        .game-container {{ background-color:#374151; border-radius:20px; padding:30px; max-width:800px; width:100%; box-shadow:0 10px 30px rgba(0,0,0,0.7); text-align:center; display:flex; flex-direction:column; gap:20px; border:2px solid #60a5fa; }}
        .title {{ font-size:2.5rem; font-weight:700; color:#60a5fa; display:flex; align-items:center; justify-content:center; text-shadow:0 0 8px rgba(96,165,250,0.8); }}
        .subtitle {{ font-size:1.3rem; color:#93c5fd; }}
        .model-badge {{ display:inline-block; background:rgba(124,58,237,0.3); border:1px solid #7c3aed; border-radius:999px; padding:3px 12px; font-size:0.8rem; color:#c4b5fd; }}
        .progress-bar-container {{ background:#4b5563; border-radius:999px; height:12px; overflow:hidden; }}
        .progress-bar-fill {{ height:100%; background:linear-gradient(90deg,#3b82f6,#8b5cf6); border-radius:999px; width:{progress_pct}%; transition:width 0.5s ease; }}
        .story-box {{ background-color:#4b5563; border-radius:15px; padding:20px; min-height:150px; display:block; font-size:1.1rem; line-height:1.6; text-align:left; overflow-y:auto; max-height:300px; border:1px solid #6b7280; animation:text-fade-in 1s ease-out; color:#e5e7eb; }}
        .ai-concept-highlight {{ display:block; margin-top:15px; padding:10px 15px; background-color:rgba(96,165,250,0.2); border-left:5px solid #60a5fa; border-radius:5px; font-weight:bold; color:#93c5fd; }}
        .choices-container {{ display:flex; flex-wrap:wrap; justify-content:center; gap:10px; margin-top:20px; }}
        .choice-button,.go-button,.save-button,.start-button {{ transition:all 0.3s ease-in-out; }}
        .choice-button:hover,.go-button:hover,.save-button:hover {{ transform:scale(1.08); box-shadow:0 5px 15px rgba(0,0,0,0.3); }}
        @keyframes pulse {{ 0% {{ box-shadow:0 0 0 0 rgba(96,165,250,0.7); }} 70% {{ box-shadow:0 0 0 15px rgba(96,165,250,0); }} 100% {{ box-shadow:0 0 0 0 rgba(96,165,250,0); }} }}
        .start-button {{ animation:pulse 2s infinite; }}
        @keyframes text-fade-in {{ 0% {{ opacity:0; }} 100% {{ opacity:1; }} }}
        .input-area {{ display:flex; gap:10px; margin-top:15px; justify-content:center; align-items:center; flex-wrap:wrap; }}
        .input-area input {{ padding:10px 15px; background-color:#4b5563; color:#e5e7eb; font-size:1rem; width:60%; border-radius:8px; border:1px solid #6b7280; }}
        .input-area input:focus {{ outline:none; border-color:#60a5fa; box-shadow:0 0 0 3px rgba(96,165,250,0.5); }}
        .control-buttons {{ display:flex; justify-content:center; gap:15px; margin-top:20px; }}
        .message-box {{ margin-top:10px; padding:15px; border-radius:10px; font-weight:bold; }}
        .hidden {{ display:none; }}
        .concepts-log {{ text-align:left; }}
    </style>

    <div class="game-container">
        <div class="title">AI Explorer!</div>
        <div class="subtitle">An Adventure in the World of Intelligent Machines</div>
        <div class="model-badge">Mistral-7B-Instruct-v0.3 &middot; 4-bit NF4 &middot; Fully Open Source</div>

        <p class="text-lg font-bold text-yellow-400">
            Score: {round(player_score)} | Turn: {turn_counter} / {TOTAL_GAME_TURNS} | Concepts: {len(covered_concepts)}
            {streak_html}
        </p>

        <div class="progress-bar-container"><div class="progress-bar-fill"></div></div>

        <div class="message-box {msg_class}">{msg_content}</div>

        <div class="story-box">{current_story_description}</div>

        <div class="choices-container">{choices_html}</div>
        {input_area_html}
        {name_input_html}
        {high_scores_html}
        {concepts_log_html}
        {sheriff_panel}

        <div class="control-buttons">
            <button class="start-button bg-purple-700 hover:bg-purple-900 text-white font-bold
                py-2 px-6 rounded-full transition duration-300 ease-in-out transform
                hover:scale-105 shadow-lg"
                onclick="google.colab.kernel.invokeFunction('start_new_game',[],{{}})">
                Start New AI Adventure
            </button>
        </div>
    </div>
    """

    display.clear_output(wait=True)
    display.display(display.HTML(html_content))


# ============================================================
# SAVE SCORE CALLBACK (synchronous)
# ============================================================
def save_score_callback(player_name):
    global player_score, info_message, error_message
    if player_name and player_name.strip():
        try:
            existing = get_high_scores()
            existing.append((player_name.strip(), round(player_score)))
            existing.sort(key=lambda x: x[1], reverse=True)
            save_high_scores_to_drive(existing)
            info_message  = f'Score for {player_name} ({round(player_score)}) saved!'
            error_message = ''
        except Exception as e:
            error_message = f'Error saving score: {e}'
            info_message  = ''
    else:
        error_message = 'Please enter a name to save your score.'
        info_message  = ''
    update_game_ui(is_loading=False)


# ============================================================
# SYNCHRONOUS WRAPPERS — original Colab callback pattern
# ============================================================
def sync_start_new_game():
    asyncio.run(start_new_game())

def sync_handle_player_action(player_action):
    asyncio.run(handle_player_action(player_action))


output.register_callback('start_new_game',       sync_start_new_game)
output.register_callback('handle_player_action', sync_handle_player_action)
output.register_callback('save_score_callback',  save_score_callback)

update_game_ui()
print(f'Game ready. H2E Sheriff active. Lambda={H2E_LAMBDA:.6f} | Mistral-7B-Instruct-v0.3 4-bit NF4')
